In [1]:
!pip -q install pypdf datasets pandas sentence-transformers faiss-cpu

import re, json, random
import numpy as np
import pandas as pd
import faiss

from datasets import load_dataset
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

random.seed(42)

encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Setup done. Encoder ready.")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup done. Encoder ready.


In [2]:
dataset = load_dataset("atta00/cpt-hcpcs-codes")
hf_df = dataset["train"].to_pandas().dropna(subset=["description"]).reset_index(drop=True)

hf_df["type"] = hf_df["type"].astype(str)
cpt_df = hf_df[hf_df["type"].str.upper() == "CPT"].reset_index(drop=True)

# Only mammograms
MAMMO_CODES = ["77067", "77063", "77065", "77066"]
cpt_df = cpt_df[cpt_df["code"].astype(str).isin(MAMMO_CODES)].reset_index(drop=True)

# Text we embed
cpt_df["embed_text"] = (
    cpt_df["code"].astype(str) + " " +
    cpt_df["title"].fillna("").astype(str) + " " +
    cpt_df["description"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

# Embed and index
cpt_embeddings = encoder.encode(
    cpt_df["embed_text"].tolist(),
    normalize_embeddings=True
).astype(np.float32)

cpt_index = faiss.IndexFlatIP(cpt_embeddings.shape[1])
cpt_index.add(cpt_embeddings)

def search_mammo_cpt(query, k=4):
    q = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = cpt_index.search(q, k)
    rows = cpt_df.iloc[idxs[0]][["code","title"]].copy()
    rows["score"] = scores[0]
    return rows.reset_index(drop=True)

print("Mammogram-only CPT matcher ready.")


Mammogram-only CPT matcher ready.


In [3]:
print(search_mammo_cpt("I need a routine screening mammogram", k=4))
print(search_mammo_cpt("diagnostic mammogram both breasts", k=4))
print(search_mammo_cpt("3D tomosynthesis screening", k=4))


    code                                              title     score
0  77067  "Breast Mammography.Diagnostic Radiology. Mamm...  0.604020
1  77066  "Breast Mammography.Diagnostic Radiology. Mamm...  0.565956
2  77065  "Breast Mammography.Diagnostic Radiology. Mamm...  0.555959
3  77063  "Breast Mammography.Diagnostic Radiology. Mamm...  0.538263
    code                                              title     score
0  77066  "Breast Mammography.Diagnostic Radiology. Mamm...  0.595124
1  77067  "Breast Mammography.Diagnostic Radiology. Mamm...  0.570037
2  77065  "Breast Mammography.Diagnostic Radiology. Mamm...  0.540239
3  77063  "Breast Mammography.Diagnostic Radiology. Mamm...  0.493659
    code                                              title     score
0  77063  "Breast Mammography.Diagnostic Radiology. Mamm...  0.523377
1  77067  "Breast Mammography.Diagnostic Radiology. Mamm...  0.285934
2  77066  "Breast Mammography.Diagnostic Radiology. Mamm...  0.262614
3  77065  "Breast Ma

In [4]:
pdf_path = "/content/evwoman.pdf"  # <-- change if your file name is different


In [5]:
pdf_path = "/content/evwoman.pdf"  # change to /content/ewwoman.pdf if needed
reader = PdfReader(pdf_path)

pages = []
for i, page in enumerate(reader.pages):
    txt = page.extract_text() or ""
    txt = re.sub(r"\s+", " ", txt).strip()
    if txt:
        pages.append({"page_num": i+1, "text": txt})

full_text = "\n".join([p["text"] for p in pages])

# simple rule extraction
screening_age_min = 40
m_age = re.search(r"average risk.*age\s+(\d{2})", full_text, flags=re.IGNORECASE)
if m_age:
    screening_age_min = int(m_age.group(1))

screening_freq_days = 365
m_freq = re.search(r"reimbursable once every\s+(\d{2,4})\s+days", full_text, flags=re.IGNORECASE)
if m_freq:
    screening_freq_days = int(m_freq.group(1))

addon_rule_present = bool(re.search(r"77063.*in addition to.*77067", full_text, flags=re.IGNORECASE))
same_day_conflict_present = bool(re.search(r"77065.*77066.*not reimbursable.*same day", full_text, flags=re.IGNORECASE))

print("Rules:", screening_age_min, screening_freq_days, addon_rule_present, same_day_conflict_present)

# Make test set
SCOPE_CODES = ["77067", "77063", "77065", "77066"]

templates = {
    "77067": ["I need a screening mammogram.", "Schedule my routine screening mammogram."],
    "77063": ["I want a 3D screening mammogram (tomosynthesis).", "Is 3D mammography covered for screening?"],
    "77065": ["Diagnostic mammography unilateral (one breast).", "I have a lump and need a diagnostic mammogram on one breast."],
    "77066": ["Diagnostic mammography bilateral (both breasts).", "Doctor ordered a diagnostic mammogram for both breasts."],
}

def build_checklist(code):
    items = []
    if code in ["77067","77063"]:
        items.append(f"Check age eligibility (>= {screening_age_min} for average risk).")
        items.append(f"Check frequency limit (1 per {screening_freq_days} days).")
        if code == "77063" and addon_rule_present:
            items.append("Check that 77063 is billed in addition to 77067.")
    if code in ["77065","77066"]:
        items.append("Diagnostic mammography requires symptoms/history/clinical indication.")
        if same_day_conflict_present:
            items.append("Check not billing 77065 and 77066 same day for same recipient.")
    return "\n- " + "\n- ".join(items)

def make_case(case_id):
    code = random.choice(SCOPE_CODES)
    attrs = dict(
        age=random.choice([35, 39, 40, 45, 55]),
        days_since_last_screening=random.choice([90, 200, 364, 365, 800, None]),
        avg_risk=True,
        symptoms_present=(random.random() < 0.6),
        history_breast_cancer=(random.random() < 0.2),
        includes_77067=True,
        same_day_codes=[]
    )

    if code == "77063" and addon_rule_present and random.random() < 0.25:
        attrs["includes_77067"] = False

    if code in ["77065","77066"] and same_day_conflict_present and random.random() < 0.2:
        attrs["same_day_codes"] = ["77065","77066"]

    user_query = random.choice(templates[code])

    label = "covered"
    if code in ["77067","77063"]:
        if attrs["age"] < screening_age_min:
            label = "not_covered"
        if attrs["days_since_last_screening"] is not None and attrs["days_since_last_screening"] < screening_freq_days:
            label = "not_covered"
        if code == "77063" and addon_rule_present and not attrs["includes_77067"]:
            label = "not_covered"

    if code in ["77065","77066"]:
        if not (attrs["symptoms_present"] or attrs["history_breast_cancer"]):
            label = "not_covered"
        if same_day_conflict_present and attrs["same_day_codes"] == ["77065","77066"]:
            label = "not_covered"

    return {
        "case_id": case_id,
        "user_query": user_query,
        "expected_cpt": code,
        "expected_label": label,
        "attrs_json": json.dumps(attrs),
        "expected_checklist": build_checklist(code)
    }

rows = [make_case(i) for i in range(1, 101)]
test_df = pd.DataFrame(rows)
test_df.to_csv("ewc_mammogram_testset_v1.csv", index=False)

print("Saved: ewc_mammogram_testset_v1.csv")
test_df.head()




Rules: 30 365 True True
Saved: ewc_mammogram_testset_v1.csv


,case_id,user_query,expected_cpt,expected_label,attrs_json,expected_checklist
0,1,I need a screening mammogram.,77067,covered,"{""age"": 35, ""days_since_last_screening"": null,...",\n- Check age eligibility (>= 30 for average r...
1,2,I need a screening mammogram.,77067,covered,"{""age"": 55, ""days_since_last_screening"": 365, ...",\n- Check age eligibility (>= 30 for average r...
2,3,Schedule my routine screening mammogram.,77067,not_covered,"{""age"": 55, ""days_since_last_screening"": 200, ...",\n- Check age eligibility (>= 30 for average r...
3,4,I want a 3D screening mammogram (tomosynthesis).,77063,covered,"{""age"": 45, ""days_since_last_screening"": 800, ...",\n- Check age eligibility (>= 30 for average r...
4,5,Diagnostic mammography bilateral (both breasts).,77066,covered,"{""age"": 40, ""days_since_last_screening"": 364, ...",\n- Diagnostic mammography requires symptoms/h...


In [6]:
sample = test_df.sample(10, random_state=1)

for _, r in sample.iterrows():
    res = search_mammo_cpt(r["user_query"], k=4)
    top_codes = res["code"].astype(str).tolist()

    print("\nQ:", r["user_query"])
    print("Expected:", r["expected_cpt"], "| Ranked:", top_codes)



Q: Is 3D mammography covered for screening?
Expected: 77063 | Ranked: ['77067', '77066', '77063', '77065']

Q: Doctor ordered a diagnostic mammogram for both breasts.
Expected: 77066 | Ranked: ['77066', '77067', '77065', '77063']

Q: Schedule my routine screening mammogram.
Expected: 77067 | Ranked: ['77067', '77066', '77063', '77065']

Q: I want a 3D screening mammogram (tomosynthesis).
Expected: 77063 | Ranked: ['77063', '77067', '77066', '77065']

Q: Doctor ordered a diagnostic mammogram for both breasts.
Expected: 77066 | Ranked: ['77066', '77067', '77065', '77063']

Q: Schedule my routine screening mammogram.
Expected: 77067 | Ranked: ['77067', '77066', '77063', '77065']

Q: I want a 3D screening mammogram (tomosynthesis).
Expected: 77063 | Ranked: ['77063', '77067', '77066', '77065']

Q: Schedule my routine screening mammogram.
Expected: 77067 | Ranked: ['77067', '77066', '77063', '77065']

Q: Schedule my routine screening mammogram.
Expected: 77067 | Ranked: ['77067', '77066', 

In [7]:
from google.colab import files
files.download("ewc_mammogram_testset_v1.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>